# Chakaria remanent mangrove productivity (uGPP)

Python + geemap + Google Earth Engine workflow for RMSP / PMSP / PMWSP sites.

**Metrics:** Mean uGPP · Temporal stability (μ/σ) · Sen slope · Mann–Kendall Tau

Run from the repository root, or `cd chakaria_mangrove_productivity` before executing cells.

In [ ]:
# Optional first-time install
# %pip install -q -r chakaria_mangrove_productivity/requirements.txt

from pathlib import Path
import os
import sys

ROOT = Path("chakaria_mangrove_productivity")
if not ROOT.exists():
    ROOT = Path(".")
os.chdir(ROOT)
sys.path.insert(0, str(Path(".").resolve()))
print("Working directory:", Path(".").resolve())

## Option A — offline demo (no GEE auth)

Generates synthetic annual uGPP, then computes metrics, statistics, and figures.

In [ ]:
!python run_pipeline.py

## Option B — real GEE extraction

Authenticate Earth Engine once, then extract GPW annual uGPP for all 30 sites.

In [ ]:
import importlib.util
from pathlib import Path

def load_script(name: str, path: str):
    spec = importlib.util.spec_from_file_location(name, Path(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

extract = load_script("extract", "scripts/01_extract_gee.py")

# Set your GCP project if required by your EE account:
GCP_PROJECT = None  # e.g. "my-ee-project"

# Uncomment to run live extraction (slow: ~30 sites × ~25 years):
# extract.initialize_ee(GCP_PROJECT)
# sites = extract.load_sites()
# df = extract.extract_timeseries_client(sites)
# df.to_csv("data/annual_ugpp.csv", index=False)
# df.head()

print("Uncomment the block above after earthengine authenticate.")

## Interactive geemap view

In [ ]:
# Requires Earth Engine auth + geemap
# extract.initialize_ee(GCP_PROJECT)
# sites = extract.load_sites()
# m = extract.build_map(sites)
# m

## Metrics, group tests, and figures

In [ ]:
import pandas as pd
from IPython.display import Image, display

!python scripts/02_temporal_metrics.py
!python scripts/03_statistics.py
!python scripts/04_figures.py

metrics = pd.read_csv("outputs/tables/site_metrics.csv")
display(metrics.head())
display(pd.read_csv("outputs/tables/kruskal_wallis.csv"))

for fig in sorted(Path("outputs/figures").glob("Figure*.png")):
    print(fig.name)
    display(Image(filename=str(fig), width=520))